Vous êtes chargé d'analyser un ensemble de documents municipaux numérisés, regroupés dans une archive  (lien vers le github ou se situe le dossier des archives : https://github.com/OpenClassrooms-Student-Center/8532116-mettez-en-place-un-rag-pour-un-llm/tree/main/inputs )  contenant divers formats tels que PDF, Word, Excel et images.

Pour faciliter l'exploitation de ces documents, vous allez utiliser l'outil open-source Docling, qui permet de convertir efficacement ces formats en fichiers Markdown ou JSON, tout en préservant la structure et le contenu des documents originaux.
Consignes

1. Installation de Docling : Assurez-vous d'avoir Python installé sur votre système. Installez Docling en exécutant la commande suivante dans votre terminal :

pip install docling

 2. Extraction de l'archive ZIP :

    Décompressez l'archive ZIP contenant les documents dans un répertoire de votre choix.

3. Conversion des documents :

    Écrivez un script Python pour convertir tous les documents du répertoire extrait en fichiers Markdown.

    Le script doit parcourir chaque fichier du répertoire, détermine son format et effectue la conversion appropriée en utilisant Docling.

In [1]:
import os

# Uninstall existing GPU-enabled torch if any, to prevent conflicts
!pip uninstall -y torch torchvision torchaudio

# Install CPU-only version of torch
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

# Reinstall transformers and docling to ensure they link to the correct torch version
!pip install --upgrade transformers
!pip install docling


Found existing installation: torch 2.11.0+cpu
Uninstalling torch-2.11.0+cpu:
  Successfully uninstalled torch-2.11.0+cpu
Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
Found existing installation: torchaudio 2.11.0+cpu
Uninstalling torchaudio-2.11.0+cpu:
  Successfully uninstalled torchaudio-2.11.0+cpu
Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached https://download.pytorch.org/whl/cpu/torch-2.11.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached https://download-r2.pytorch.org/whl/cpu/torchvision-0.26.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached https://download-r2.pytorch.org/whl/cpu/torchaudio-2.11.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
Using cached https://download.pytorch.org/whl/cpu/torch-2.11.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl (190.3 MB)
Using cached https://download-r2

In [2]:
# Cloner le repo directement depuis GitHub
!git clone https://github.com/OpenClassrooms-Student-Center/8532116-mettez-en-place-un-rag-pour-un-llm.git

fatal: destination path '8532116-mettez-en-place-un-rag-pour-un-llm' already exists and is not an empty directory.


In [3]:
#Verifier la structure des fichiers téléchargés

import os

input_dir = "8532116-mettez-en-place-un-rag-pour-un-llm/inputs"

print("📂 Structure du répertoire inputs :\n")
for root, dirs, files in os.walk(input_dir):
    # Niveau d'indentation
    level = root.replace(input_dir, '').count(os.sep)
    indent = '  ' * level
    folder_name = os.path.basename(root)
    print(f"{indent}📁 {folder_name}/")
    for f in files:
        ext = os.path.splitext(f)[1].lower()
        print(f"{indent}  📄 {f}  [{ext}]")


📂 Structure du répertoire inputs :

📁 inputs/
  📁 communication/
    📄 logo_triffouillis.webp  [.webp]
    📄 voeux2025_trifouillis.wav  [.wav]
    📄 Acceuil_affichage_Mairie de Triffouillis sur Loire.pdf  [.pdf]
  📁 budget/
    📄 budget_2024.csv  [.csv]
  📁 intances/
    📄 PV_15112023.pdf  [.pdf]
    📄 RÈGLEMENT MUNICIPAL.pdf  [.pdf]
    📄 Bulletin Municipal – Intervention Technique et Sécurité Routière.pdf  [.pdf]
    📄 Bulletin Municipal – Marché de Noël 2023.pdf  [.pdf]
    📄 PV_20122023.pdf  [.pdf]
    📄 Bulletin Municipal – Réunion Publique.pdf  [.pdf]
    📄 PV_05102023.pdf  [.pdf]
  📁 projets/
    📄 projets_2024.csv  [.csv]
    📄 Installation d'un système de surveillance urbaine.pdf  [.pdf]
    📄 Réaménagement de la place du Marché.pdf  [.pdf]
    📄 Développement de nouvelles pistes cyclables et zones piétonnes (2026).pdf  [.pdf]
    📄 Projet de Création du Centre Culturel Innovant.pdf  [.pdf]
    📄 Suivi du projet de Centre Culturel Innovant.pdf  [.pdf]
    📄 Projet de Modernisa

In [4]:
#Script de conversion Docling

import os
from docling.document_converter import DocumentConverter

# --- Configuration ---
input_dir = "8532116-mettez-en-place-un-rag-pour-un-llm/inputs"
output_dir = "documents_markdown"
os.makedirs(output_dir, exist_ok=True)

# Extensions supportées par Docling
SUPPORTED_EXTENSIONS = {
    '.pdf', '.docx', '.pptx', '.xlsx',
    '.html', '.htm',
    '.png', '.jpg', '.jpeg', '.tiff', '.tif', '.bmp',
    '.csv',
    '.txt', '.text',
}

# Note : .webp et .wav peuvent nécessiter des dépendances supplémentaires
# On les tente quand même, Docling gère le WAV (ASR) et certaines images

# --- Initialiser le convertisseur ---
converter = DocumentConverter()

# --- Compteurs ---
converted_count = 0
error_count = 0
skipped_count = 0
errors_detail = []

# --- Parcourir tous les fichiers ---
for root, dirs, files_list in os.walk(input_dir):
    for filename in sorted(files_list):
        filepath = os.path.join(root, filename)
        file_ext = os.path.splitext(filename)[1].lower()

        # Vérifier si le format est supporté
        if file_ext not in SUPPORTED_EXTENSIONS:
            print(f"⏭️  Ignoré (format {file_ext}) : {filename}")
            skipped_count += 1
            continue

        # Déterminer le sous-dossier d'origine pour organiser la sortie
        relative_path = os.path.relpath(root, input_dir)
        output_subdir = os.path.join(output_dir, relative_path)
        os.makedirs(output_subdir, exist_ok=True)

        print(f"🔄 Conversion : {os.path.join(relative_path, filename)} ...")

        try:
            # Conversion
            result = converter.convert(filepath)

            # Export Markdown
            markdown_content = result.document.export_to_markdown()

            # Sauvegarder le fichier .md
            md_filename = os.path.splitext(filename)[0] + ".md"
            md_filepath = os.path.join(output_subdir, md_filename)

            with open(md_filepath, "w", encoding="utf-8") as md_file:
                md_file.write(markdown_content)

            print(f"   ✅ → {md_filepath}")
            converted_count += 1

        except Exception as e:
            print(f"   ❌ Erreur : {e}")
            error_count += 1
            errors_detail.append((filename, str(e)))

# --- Résumé ---
print("\n" + "=" * 60)
print(f"📊 RÉSUMÉ DE LA CONVERSION")
print(f"=" * 60)
print(f"  ✅ Convertis avec succès : {converted_count}")
print(f"  ❌ Erreurs               : {error_count}")
print(f"  ⏭️  Ignorés               : {skipped_count}")
print(f"  📁 Total parcouru        : {converted_count + error_count + skipped_count}")

if errors_detail:
    print(f"\n📋 Détail des erreurs :")
    for fname, err in errors_detail:
        print(f"   - {fname} : {err}")


🔄 Conversion : communication/Acceuil_affichage_Mairie de Triffouillis sur Loire.pdf ...


[INFO] 2026-03-27 22:01:59,396 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-27 22:01:59,443 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-27 22:01:59,445 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-27 22:01:59,736 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-27 22:01:59,749 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-27 22:01:59,750 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-27 22:01:59,831 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-27 22:01:59,910 [RapidOCR] download_file.py:60: File exists and is valid: /usr/loc

   ✅ → documents_markdown/communication/Acceuil_affichage_Mairie de Triffouillis sur Loire.md
⏭️  Ignoré (format .webp) : logo_triffouillis.webp
⏭️  Ignoré (format .wav) : voeux2025_trifouillis.wav
🔄 Conversion : budget/budget_2024.csv ...
   ✅ → documents_markdown/budget/budget_2024.md
🔄 Conversion : intances/Bulletin Municipal – Intervention Technique et Sécurité Routière.pdf ...
   ✅ → documents_markdown/intances/Bulletin Municipal – Intervention Technique et Sécurité Routière.md
🔄 Conversion : intances/Bulletin Municipal – Marché de Noël 2023.pdf ...
   ✅ → documents_markdown/intances/Bulletin Municipal – Marché de Noël 2023.md
🔄 Conversion : intances/Bulletin Municipal – Réunion Publique.pdf ...
   ✅ → documents_markdown/intances/Bulletin Municipal – Réunion Publique.md
🔄 Conversion : intances/PV_05102023.pdf ...
   ✅ → documents_markdown/intances/PV_05102023.md
🔄 Conversion : intances/PV_15112023.pdf ...
   ✅ → documents_markdown/intances/PV_15112023.md
🔄 Conversion : intances/PV

In [5]:
#Visualiser les résultats
# Afficher un aperçu de chaque fichier Markdown généré
for root, dirs, files_list in os.walk(output_dir):
    for md_file in sorted(files_list):
        if md_file.endswith(".md"):
            filepath = os.path.join(root, md_file)
            with open(filepath, "r", encoding="utf-8") as f:
                content = f.read()

            relative = os.path.relpath(filepath, output_dir)
            print(f"\n{'='*60}")
            print(f"📄 {relative}  ({len(content)} caractères)")
            print(f"{'='*60}")
            print(content[:500])
            if len(content) > 500:
                print("...")



📄 communication/Acceuil_affichage_Mairie de Triffouillis sur Loire.md  (5935 caractères)
## Mairie de Triffouillis sur Loire Avis d'Information et Horaires d'Ouverture

## I. Message d'Accueil

## Chers

<!-- image -->

concitoyens,

Nous avons le plaisir de vous accueillir sous l'égide de la bonne humeur et du sérieux nécessaire à l'exercice de nos fonctions. La Mairie de Triffouillis sur Loire vous invite à consulter les informations ci-après afin de mieux connaître nos services et modalités de fonctionnement.

Nous vous présentons, en tête d'affiche, notre maire officieux et atta
...

📄 budget/budget_2024.md  (939 caractères)
| Catégorie           |   Budget Alloué (en €) |   Dépenses Réelles (en €) |   Solde (en €) |
|---------------------|------------------------|---------------------------|----------------|
| Administration      |                 500000 |                    480000 |          20000 |
| Voirie              |                 750000 |                    700000 |    

Comprendre le Chunking
Pourquoi découper les documents ?

Imagine que tu poses une question précise : "Quel est le budget prévu pour l'éclairage public ?". Si tu envoies au LLM un document PDF entier de 15 pages, il va se noyer dans l'information. Le chunking consiste à découper tes documents en morceaux (chunks) de taille raisonnable, pour que lors de la recherche, on puisse retrouver et envoyer au LLM uniquement les passages pertinents.
Le dilemme fondamental : taille des chunks

C'est un vrai équilibre à trouver. Des chunks trop petits (ex: 100 tokens) captureront des phrases isolées, hors contexte, et le LLM aura du mal à comprendre de quoi il s'agit. Des chunks trop grands (ex: 5000 tokens) contiendront trop d'informations mélangées, ce qui dilue la pertinence lors de la recherche vectorielle et consomme du contexte inutilement. En pratique, on vise généralement entre 500 et 1500 caractères selon les cas d'usage.
L'overlap (chevauchement) : l'astuce clé

Quand tu coupes un texte en morceaux, tu risques de couper une idée en plein milieu. L'overlap résout ce problème : chaque chunk reprend les derniers caractères du chunk précédent. Par exemple, avec des chunks de 1000 caractères et un overlap de 200 :
```
Chunk 1 : caractères 0    → 1000
Chunk 2 : caractères 800  → 1800    (200 de chevauchement)
Chunk 3 : caractères 1600 → 2600    (200 de chevauchement)
```

Cela garantit qu'une phrase coupée à la frontière d'un chunk sera complète dans le chunk suivant.
Les stratégies de chunking

Il existe plusieurs approches. Le découpage par taille fixe (RecursiveCharacterTextSplitter) est le plus courant : on coupe à une taille donnée en essayant de respecter les frontières naturelles du texte (paragraphes, puis phrases, puis mots). Le découpage sémantique regroupe les passages par sens, mais il est plus coûteux. Le découpage structurel exploite la structure du document (titres, sections), ce qui est parfaitement adapté à nos fichiers Markdown puisque Docling a préservé les titres avec #, ##, etc.

Pour cet exercice, on va utiliser LangChain avec deux approches complémentaires : le RecursiveCharacterTextSplitter pour le découpage par taille, et le MarkdownHeaderTextSplitter qui exploite la structure Markdown de nos documents.

In [6]:
#Installation de LangChain
!pip install langchain langchain-text-splitters


In [7]:
# Charger tout les fichiers Markdown
import os

output_dir = "documents_markdown"

documents = []

for root, dirs, files_list in os.walk(output_dir):
    for filename in sorted(files_list):
        if filename.endswith(".md"):
            filepath = os.path.join(root, filename)
            with open(filepath, "r", encoding="utf-8") as f:
                content = f.read()

            # On garde la catégorie (nom du sous-dossier) comme métadonnée
            category = os.path.relpath(root, output_dir)

            documents.append({
                "content": content,
                "metadata": {
                    "source": filename,
                    "category": category,
                    "filepath": filepath
                }
            })

print(f"📄 {len(documents)} documents Markdown chargés\n")
for doc in documents:
    print(f"  [{doc['metadata']['category']}] {doc['metadata']['source']} "
          f"({len(doc['content'])} caractères)")


📄 28 documents Markdown chargés

  [communication] Acceuil_affichage_Mairie de Triffouillis sur Loire.md (5935 caractères)
  [budget] budget_2024.md (939 caractères)
  [intances] Bulletin Municipal – Intervention Technique et Sécurité Routière.md (1460 caractères)
  [intances] Bulletin Municipal – Marché de Noël 2023.md (1287 caractères)
  [intances] Bulletin Municipal – Réunion Publique.md (1292 caractères)
  [intances] PV_05102023.md (1742 caractères)
  [intances] PV_15112023.md (1837 caractères)
  [intances] PV_20122023.md (1815 caractères)
  [intances] RÈGLEMENT MUNICIPAL.md (25543 caractères)
  [projets] Développement de nouvelles pistes cyclables et zones piétonnes (2026).md (6428 caractères)
  [projets] Installation d'un système de surveillance urbaine.md (10510 caractères)
  [projets] Installation de panneaux photovoltaïques sur les bâtiments municipaux.md (8016 caractères)
  [projets] Projet d'Amélioration des Espaces Verts.md (3749 caractères)
  [projets] Projet de Création d

In [8]:
# Chunking avec RecursiveCharacterTextSplitter

from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- Configuration du splitter ---
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,        # Taille max d'un chunk en caractères
    chunk_overlap=200,      # Chevauchement entre chunks consécutifs
    length_function=len,
    separators=[
        "\n## ",   # D'abord couper aux titres de niveau 2
        "\n### ",  # Puis aux titres de niveau 3
        "\n\n",    # Puis aux doubles sauts de ligne (paragraphes)
        "\n",      # Puis aux sauts de ligne simples
        ". ",      # Puis aux phrases
        " ",       # Puis aux mots
        ""         # En dernier recours, caractère par caractère
    ]
)

# --- Découper tous les documents ---
all_chunks = []

for doc in documents:
    chunks = text_splitter.create_documents(
        texts=[doc["content"]],
        metadatas=[doc["metadata"]]
    )
    all_chunks.extend(chunks)

print(f"✂️  {len(all_chunks)} chunks créés à partir de {len(documents)} documents\n")

# --- Statistiques ---
chunk_sizes = [len(c.page_content) for c in all_chunks]
print(f"📊 Statistiques des chunks :")
print(f"   Taille min     : {min(chunk_sizes)} caractères")
print(f"   Taille max     : {max(chunk_sizes)} caractères")
print(f"   Taille moyenne : {sum(chunk_sizes) // len(chunk_sizes)} caractères")


✂️  195 chunks créés à partir de 28 documents

📊 Statistiques des chunks :
   Taille min     : 12 caractères
   Taille max     : 999 caractères
   Taille moyenne : 664 caractères


In [9]:
# Visualiser quelques Chunks
# Afficher les 5 premiers chunks pour vérifier la qualité
for i, chunk in enumerate(all_chunks[:5]):
    print(f"\n{'='*60}")
    print(f"📦 Chunk {i+1}")
    print(f"   Source   : {chunk.metadata['source']}")
    print(f"   Catégorie: {chunk.metadata['category']}")
    print(f"   Taille   : {len(chunk.page_content)} caractères")
    print(f"{'='*60}")
    print(chunk.page_content)



📦 Chunk 1
   Source   : Acceuil_affichage_Mairie de Triffouillis sur Loire.md
   Catégorie: communication
   Taille   : 737 caractères
## Mairie de Triffouillis sur Loire Avis d'Information et Horaires d'Ouverture

## I. Message d'Accueil

## Chers

<!-- image -->

concitoyens,

Nous avons le plaisir de vous accueillir sous l'égide de la bonne humeur et du sérieux nécessaire à l'exercice de nos fonctions. La Mairie de Triffouillis sur Loire vous invite à consulter les informations ci-après afin de mieux connaître nos services et modalités de fonctionnement.

Nous vous présentons, en tête d'affiche, notre maire officieux et attachant : Madame Pétillante Rigolade , qui, avec un sourire communicatif, veille à la bonne marche de notre commune. Son engagement, conjugué à l'expertise de nos équipes, vous assure une prise en charge rapide et efficace de vos demandes.

📦 Chunk 2
   Source   : Acceuil_affichage_Mairie de Triffouillis sur Loire.md
   Catégorie: communication
   Taille   : 323 c

In [10]:
# Analyse de la distribution par documents
from collections import Counter

# Combien de chunks par document source ?
chunks_per_doc = Counter(c.metadata['source'] for c in all_chunks)

print(f"📊 Répartition des chunks par document :\n")
for source, count in chunks_per_doc.most_common():
    print(f"   {count:3d} chunks ← {source}")

# Combien de chunks par catégorie ?
print(f"\n📊 Répartition par catégorie :\n")
chunks_per_cat = Counter(c.metadata['category'] for c in all_chunks)
for cat, count in chunks_per_cat.most_common():
    print(f"   {count:3d} chunks ← {cat}")


📊 Répartition des chunks par document :

    44 chunks ← RÈGLEMENT MUNICIPAL.md
    16 chunks ← Installation de panneaux photovoltaïques sur les bâtiments municipaux.md
    16 chunks ← Projet de Création du Centre Culturel Innovant.md
    14 chunks ← Installation d'un système de surveillance urbaine.md
    13 chunks ← Projet de Modernisation de l eclairage public.md
    13 chunks ← Projet de Rénovation de la Voirie Centrale.md
    12 chunks ← Réaménagement de la place du Marché.md
    10 chunks ← Développement de nouvelles pistes cyclables et zones piétonnes (2026).md
     9 chunks ← Acceuil_affichage_Mairie de Triffouillis sur Loire.md
     8 chunks ← Suivi du projet de Centre Culturel Innovant.md
     6 chunks ← Projet d'Amélioration des Espaces Verts.md
     4 chunks ← PV_15112023.md
     4 chunks ← PV_20122023.md
     4 chunks ← projets_2024.md
     3 chunks ← Bulletin Municipal – Intervention Technique et Sécurité Routière.md
     3 chunks ← PV_05102023.md
     3 chunks ← marche_n

##Vectorisation
Comprendre la Vectorisation (Embeddings)
C'est quoi un embedding concrètement ?

Un embedding, c'est la traduction d'un texte en une liste de nombres (un vecteur). Par exemple, le chunk "Rénovation de l'éclairage public rue des Érables" pourrait devenir quelque chose comme [0.12, -0.45, 0.78, 0.33, ..., -0.21] — un vecteur de plusieurs centaines de dimensions.

Ce qui est magique, c'est que ces nombres capturent le sens du texte. Ainsi, "Modernisation des lampadaires avenue du Centre" aura un vecteur très similaire au précédent (même si les mots sont différents), parce que sémantiquement c'est proche. En revanche, "Budget du festival de musique" aura un vecteur éloigné, car le sujet est différent.
Comment ça marche la recherche ensuite ?

Quand un utilisateur pose une question, le RAG fait trois choses. D'abord, il transforme la question en vecteur avec le même modèle d'embedding. Ensuite, il compare ce vecteur avec tous les vecteurs des chunks stockés, en calculant une distance (souvent la similarité cosinus). Enfin, il récupère les chunks les plus proches et les envoie au LLM comme contexte pour générer la réponse.
Le modèle qu'on va utiliser : all-MiniLM-L6-v2

C'est un modèle open-source de la bibliothèque sentence-transformers, très populaire et léger. Il produit des vecteurs de 384 dimensions, ce qui est un bon compromis entre qualité et performance. Pour un projet de production, on choisirait un modèle plus puissant (comme Gemini Embedding, ou e5-large), mais pour apprendre, il est parfait.
La base vectorielle : ChromaDB

Une fois les vecteurs calculés, il faut les stocker quelque part pour pouvoir chercher dedans. ChromaDB est une base vectorielle open-source, légère, qui tourne en local. Elle stocke à la fois les vecteurs, le texte original et les métadonnées de chaque chunk.

In [11]:
!pip install sentence-transformers chromadb


In [12]:
# Créer les Embeddings et stocker dans ChromaDB

from sentence_transformers import SentenceTransformer
import chromadb
import time

# --- 1. Charger le modèle d'embedding ---
print("🔄 Chargement du modèle d'embedding...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Modèle chargé !\n")

# --- 2. Préparer les données des chunks ---
# (all_chunks vient de l'étape précédente)
texts = [chunk.page_content for chunk in all_chunks]
metadatas = [chunk.metadata for chunk in all_chunks]
ids = [f"chunk_{i}" for i in range(len(all_chunks))]

# --- 3. Générer les embeddings ---
print(f"🔄 Vectorisation de {len(texts)} chunks...")
start_time = time.time()

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32
)

elapsed = time.time() - start_time
print(f"✅ Vectorisation terminée en {elapsed:.1f} secondes")
print(f"   Dimension des vecteurs : {embeddings.shape[1]}")
print(f"   Shape totale : {embeddings.shape}\n")

# --- 4. Stocker dans ChromaDB ---
print("🔄 Stockage dans ChromaDB...")

# Créer le client et la collection
chroma_client = chromadb.Client()  # En mémoire (pour Colab)

# Supprimer la collection si elle existe déjà (utile en cas de re-run)
try:
    chroma_client.delete_collection("documents_municipaux")
except:
    pass

collection = chroma_client.create_collection(
    name="documents_municipaux",
    metadata={"description": "Documents municipaux de Triffouillis sur Loire"}
)

# Ajouter les chunks avec leurs embeddings et métadonnées
collection.add(
    ids=ids,
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=metadatas
)

print(f"✅ {collection.count()} chunks stockés dans ChromaDB !")


🔄 Chargement du modèle d'embedding...
✅ Modèle chargé !

🔄 Vectorisation de 195 chunks...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Vectorisation terminée en 37.8 secondes
   Dimension des vecteurs : 384
   Shape totale : (195, 384)

🔄 Stockage dans ChromaDB...
✅ 195 chunks stockés dans ChromaDB !


In [13]:
# Vérifier avec un premier test de recherche

# --- Testons la recherche sémantique ! ---

query = "Quel est le budget prévu pour l'éclairage public ?"

# Vectoriser la question
query_embedding = embedding_model.encode([query]).tolist()

# Rechercher les 5 chunks les plus proches
results = collection.query(
    query_embeddings=query_embedding,
    n_results=5
)

print(f"🔍 Question : \"{query}\"\n")
print(f"{'='*60}")
print(f"📊 Top 5 des chunks les plus pertinents :")
print(f"{'='*60}\n")

for i in range(len(results['documents'][0])):
    doc = results['documents'][0][i]
    meta = results['metadatas'][0][i]
    distance = results['distances'][0][i]

    # Plus la distance est petite, plus c'est pertinent
    print(f"📦 Résultat {i+1} (distance: {distance:.4f})")
    print(f"   Source    : {meta['source']}")
    print(f"   Catégorie : {meta['category']}")
    print(f"   Extrait   : {doc[:300]}...")
    print()




🔍 Question : "Quel est le budget prévu pour l'éclairage public ?"

📊 Top 5 des chunks les plus pertinents :

📦 Résultat 1 (distance: 0.7856)
   Source    : Suivi du projet de Centre Culturel Innovant.md
   Catégorie : projets
   Extrait   : ## 3. Suivi Budgétaire

- Budget Initial Global (voté en 2023) : 3 500 000 € HT
- Dépenses Engagées au 30/06/2025 : 2 150 000 € HT (soit 61.4% du budget initial)
- Analyse : Les dépenses sont globalement conformes au prévisionnel. Une légère augmentation (+3%) a été constatée sur le lot "Fondations ...

📦 Résultat 2 (distance: 0.7973)
   Source    : Développement de nouvelles pistes cyclables et zones piétonnes (2026).md
   Catégorie : projets
   Extrait   : ## 4. Budget Prévisionnel Global 2026

Le budget total estimé pour la mise en œuvre de ce plan d'action en 2026 s'élève à 395 000 € HT . Ce financement sera assuré par :

- Budget municipal d'investissement : [Montant à définir, ex : 295 000 €]
- Demandes de subventions (Département, Région, Éta

In [14]:
# Tester plusieurs questions

# Testons avec différentes questions pour voir la qualité
questions = [
    "Quand a lieu le marché de Noël ?",
    "Comment signaler un problème d'éclairage public ?",
    "Quels sont les projets de pistes cyclables ?",
    "Quels sont les horaires d'ouverture de la mairie ?",
    "Quel est le règlement municipal concernant le bruit ?",
]

for query in questions:
    query_embedding = embedding_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=3  # Top 3 seulement
    )

    print(f"\n{'='*60}")
    print(f"🔍 \"{query}\"")
    print(f"{'='*60}")

    for i in range(len(results['documents'][0])):
        meta = results['metadatas'][0][i]
        distance = results['distances'][0][i]
        print(f"   {i+1}. [{meta['category']}] {meta['source']} "
              f"(distance: {distance:.4f})")



🔍 "Quand a lieu le marché de Noël ?"
   1. [intances] Bulletin Municipal – Marché de Noël 2023.md (distance: 0.7780)
   2. [intances] Bulletin Municipal – Marché de Noël 2023.md (distance: 0.8644)
   3. [projets] Réaménagement de la place du Marché.md (distance: 0.9352)

🔍 "Comment signaler un problème d'éclairage public ?"
   1. [demandes citoyennes] Signalement et demande d'intervention pour un éclairage public défectueux.md (distance: 0.6257)
   2. [projets] Projet de Modernisation de l eclairage public.md (distance: 0.8492)
   3. [intances] RÈGLEMENT MUNICIPAL.md (distance: 0.9364)

🔍 "Quels sont les projets de pistes cyclables ?"
   1. [projets] Développement de nouvelles pistes cyclables et zones piétonnes (2026).md (distance: 0.7812)
   2. [projets] projets_2024.md (distance: 0.9091)
   3. [projets] Développement de nouvelles pistes cyclables et zones piétonnes (2026).md (distance: 0.9373)

🔍 "Quels sont les horaires d'ouverture de la mairie ?"
   1. [communication] Acceuil_aff

In [15]:
# Explorer un vecteur pour comprendre
import numpy as np

# Visualiser concrètement à quoi ressemble un embedding
exemple_text = all_chunks[0].page_content[:100]
exemple_embedding = embeddings[0]

print(f"📝 Texte : \"{exemple_text}...\"\n")
print(f"📐 Vecteur ({len(exemple_embedding)} dimensions) :")
print(f"   Premiers 10 éléments : {exemple_embedding[:10].round(4)}")
print(f"   Min : {exemple_embedding.min():.4f}")
print(f"   Max : {exemple_embedding.max():.4f}")
print(f"   Moyenne : {exemple_embedding.mean():.4f}")

# Montrer la similarité entre deux textes proches vs éloignés
text_a = "rénovation de l'éclairage public"
text_b = "modernisation des lampadaires de la ville"
text_c = "festival de musique et animations"

emb_a, emb_b, emb_c = embedding_model.encode([text_a, text_b, text_c])

# Similarité cosinus
from numpy.linalg import norm
def cosine_sim(v1, v2):
    return np.dot(v1, v2) / (norm(v1) * norm(v2))

print(f"\n🔬 Démonstration de similarité sémantique :")
print(f"   \"{text_a}\"")
print(f"   vs \"{text_b}\"")
print(f"   → Similarité : {cosine_sim(emb_a, emb_b):.4f}  (proches = thème similaire)\n")

print(f"   \"{text_a}\"")
print(f"   vs \"{text_c}\"")
print(f"   → Similarité : {cosine_sim(emb_a, emb_c):.4f}  (éloignés = thèmes différents)")


📝 Texte : "## Mairie de Triffouillis sur Loire Avis d'Information et Horaires d'Ouverture

## I. Message d'Accu..."

📐 Vecteur (384 dimensions) :
   Premiers 10 éléments : [-0.0469  0.0123 -0.0347 -0.0585 -0.0097 -0.0068  0.0413  0.0326  0.0345
  0.0183]
   Min : -0.1440
   Max : 0.1286
   Moyenne : -0.0025

🔬 Démonstration de similarité sémantique :
   "rénovation de l'éclairage public"
   vs "modernisation des lampadaires de la ville"
   → Similarité : 0.5041  (proches = thème similaire)

   "rénovation de l'éclairage public"
   vs "festival de musique et animations"
   → Similarité : 0.3800  (éloignés = thèmes différents)


## Ce qu'il faut retenir ici

La dimension du vecteur (384 pour ce modèle) représente le nombre de "caractéristiques sémantiques" que le modèle capture. Plus il y a de dimensions, plus la représentation est riche, mais plus c'est coûteux en mémoire et en calcul.

La distance dans les résultats ChromaDB mesure l'éloignement sémantique. Plus elle est proche de 0, plus le chunk est pertinent par rapport à la question. Une distance de 0.5 c'est bon, au-delà de 1.5 c'est probablement du bruit.

ChromaDB ici tourne en mémoire (pas persisté sur disque). C'est parfait pour Colab, mais dans un vrai projet tu utiliserais le mode persistant (chromadb.PersistentClient(path="./chroma_db")), ou une base vectorielle plus robuste comme Pinecone, Weaviate, ou Qdrant.

##Analyse honnête des résultats
### Ce qui marche bien

Certaines requêtes donnent d'excellents résultats. "Comment signaler un problème d'éclairage public ?" retrouve en premier le bon document avec une distance de 0.6257, c'est le meilleur score de tout le lot. "Quand a lieu le marché de Noël ?" tape directement dans le bon bulletin municipal. "Quels sont les projets de pistes cyclables ?" pointe vers le bon document aussi. La recherche sémantique fonctionne, le système comprend globalement le sens des questions.

### Ce qui pose problème

Regardons la question "Quel est le budget prévu pour l'éclairage public ?". Le document Projet de Modernisation de l'éclairage public.md n'arrive qu'en 3ème position (distance 0.7993), derrière le Centre Culturel et les pistes cyclables. Pourquoi ? Parce que les résultats 1 et 2 contiennent le mot "budget" de manière proéminente dans leur texte, et le modèle all-MiniLM-L6-v2 s'est accroché à ça. Il a priorisé le mot "budget" plutôt que la combinaison "budget + éclairage public".

Pour le règlement sur le bruit, le document RÈGLEMENT MUNICIPAL.md n'arrive qu'en 2ème position, derrière un projet d'éclairage. C'est clairement un faux positif.
Pourquoi ces résultats mitigés ?

Trois raisons principales expliquent ça. Premièrement, le modèle all-MiniLM-L6-v2 est petit et généraliste. Il fait 80 Mo, produit des vecteurs de seulement 384 dimensions, et il est entraîné en anglais principalement. Or nos documents sont en français, ce qui réduit sa performance sémantique. Deuxièmement, les distances sont globalement hautes (entre 0.6 et 1.0). Dans un bon système, on voudrait voir des distances inférieures à 0.5 pour les meilleurs résultats. Troisièmement, le chunking uniforme ne distingue pas un titre d'un paragraphe de détail, tout a le même poids.

## Comment améliorer ? Deux leviers concrets

### Levier 1 — Un meilleur modèle d'embedding (multilingue)

Remplaçons le modèle par un qui comprend vraiment le français :

In [16]:
# --- Modèle multilingue, bien meilleur pour le français ---
# Remplace 'all-MiniLM-L6-v2' (anglais, 384 dim)
# par 'paraphrase-multilingual-MiniLM-L12-v2' (50+ langues, 384 dim)

print("🔄 Chargement du modèle multilingue...")
embedding_model_v2 = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("✅ Modèle multilingue chargé !\n")

# --- Re-vectoriser tous les chunks ---
print(f"🔄 Re-vectorisation de {len(texts)} chunks...")
start_time = time.time()

embeddings_v2 = embedding_model_v2.encode(
    texts,
    show_progress_bar=True,
    batch_size=32
)

elapsed = time.time() - start_time
print(f"✅ Terminé en {elapsed:.1f}s — Shape : {embeddings_v2.shape}")

# --- Recréer la collection ChromaDB ---
try:
    chroma_client.delete_collection("documents_municipaux_v2")
except:
    pass

collection_v2 = chroma_client.create_collection(
    name="documents_municipaux_v2",
    metadata={"description": "V2 - Modèle multilingue"}
)

collection_v2.add(
    ids=ids,
    embeddings=embeddings_v2.tolist(),
    documents=texts,
    metadatas=metadatas
)

print(f"✅ {collection_v2.count()} chunks stockés dans la collection V2 !")


🔄 Chargement du modèle multilingue...
✅ Modèle multilingue chargé !

🔄 Re-vectorisation de 195 chunks...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Terminé en 40.6s — Shape : (195, 384)
✅ 195 chunks stockés dans la collection V2 !


## Levier 2 — Comparer V1 vs V2 côte à côte

In [17]:
# --- Comparaison directe des deux modèles ---

questions = [
    "Quel est le budget prévu pour l'éclairage public ?",
    "Quand a lieu le marché de Noël ?",
    "Comment signaler un problème d'éclairage public ?",
    "Quels sont les projets de pistes cyclables ?",
    "Quels sont les horaires d'ouverture de la mairie ?",
    "Quel est le règlement municipal concernant le bruit ?",
]

for query in questions:
    print(f"\n{'='*70}")
    print(f"🔍 \"{query}\"")
    print(f"{'='*70}")

    # V1 - Modèle anglais
    emb_v1 = embedding_model.encode([query]).tolist()
    res_v1 = collection.query(query_embeddings=emb_v1, n_results=3)

    # V2 - Modèle multilingue
    emb_v2 = embedding_model_v2.encode([query]).tolist()
    res_v2 = collection_v2.query(query_embeddings=emb_v2, n_results=3)

    print(f"\n  📕 V1 (all-MiniLM-L6-v2 — anglais) :")
    for i in range(3):
        meta = res_v1['metadatas'][0][i]
        dist = res_v1['distances'][0][i]
        print(f"     {i+1}. [{meta['category']}] {meta['source'][:60]} "
              f"(dist: {dist:.4f})")

    print(f"\n  📗 V2 (multilingual-MiniLM-L12 — multilingue) :")
    for i in range(3):
        meta = res_v2['metadatas'][0][i]
        dist = res_v2['distances'][0][i]
        print(f"     {i+1}. [{meta['category']}] {meta['source'][:60]} "
              f"(dist: {dist:.4f})")



🔍 "Quel est le budget prévu pour l'éclairage public ?"

  📕 V1 (all-MiniLM-L6-v2 — anglais) :
     1. [projets] Suivi du projet de Centre Culturel Innovant.md (dist: 0.7856)
     2. [projets] Développement de nouvelles pistes cyclables et zones piétonn (dist: 0.7973)
     3. [projets] Projet de Modernisation de l eclairage public.md (dist: 0.7993)

  📗 V2 (multilingual-MiniLM-L12 — multilingue) :
     1. [projets] Projet de Modernisation de l eclairage public.md (dist: 10.6913)
     2. [projets] Projet de Modernisation de l eclairage public.md (dist: 11.7923)
     3. [projets] Projet de Modernisation de l eclairage public.md (dist: 12.4179)

🔍 "Quand a lieu le marché de Noël ?"

  📕 V1 (all-MiniLM-L6-v2 — anglais) :
     1. [intances] Bulletin Municipal – Marché de Noël 2023.md (dist: 0.7780)
     2. [intances] Bulletin Municipal – Marché de Noël 2023.md (dist: 0.8644)
     3. [projets] Réaménagement de la place du Marché.md (dist: 0.9352)

  📗 V2 (multilingual-MiniLM-L12 — multilingu

## Levier 3 (bonus) — Démonstration de la similarité FR

In [18]:
# Montrer pourquoi le multilingue est meilleur pour le français
phrases = [
    "budget de l'éclairage public",
    "financement des lampadaires de la commune",
    "marché de Noël de la ville",
]

print("🔬 Comparaison des similarités (français) :\n")
print(f"   Phrases : ")
for i, p in enumerate(phrases):
    print(f"     [{i+1}] \"{p}\"")

emb_v1 = embedding_model.encode(phrases)
emb_v2 = embedding_model_v2.encode(phrases)

from numpy.linalg import norm
import numpy as np
def cosine_sim(v1, v2):
    return np.dot(v1, v2) / (norm(v1) * norm(v2))

print(f"\n   Similarité [1] vs [2] (même thème, mots différents) :")
print(f"     V1 (anglais)     : {cosine_sim(emb_v1[0], emb_v1[1]):.4f}")
print(f"     V2 (multilingue) : {cosine_sim(emb_v2[0], emb_v2[1]):.4f}")

print(f"\n   Similarité [1] vs [3] (thèmes différents) :")
print(f"     V1 (anglais)     : {cosine_sim(emb_v1[0], emb_v1[2]):.4f}")
print(f"     V2 (multilingue) : {cosine_sim(emb_v2[0], emb_v2[2]):.4f}")

print(f"\n   → ")
print(f"     ")


🔬 Comparaison des similarités (français) :

   Phrases : 
     [1] "budget de l'éclairage public"
     [2] "financement des lampadaires de la commune"
     [3] "marché de Noël de la ville"

   Similarité [1] vs [2] (même thème, mots différents) :
     V1 (anglais)     : 0.3714
     V2 (multilingue) : 0.8130

   Similarité [1] vs [3] (thèmes différents) :
     V1 (anglais)     : 0.3868
     V2 (multilingue) : 0.2998

   → 
     



## Ce qu'il faut retenir

Le choix du modèle d'embedding est critique dans un RAG. Un modèle entraîné principalement en anglais va mal comprendre les nuances sémantiques du français. C'est comme demander à quelqu'un qui parle un peu français de classer des documents administratifs français : il se débrouillera, mais il fera des erreurs qu'un francophone natif ne ferait pas.

Dans un vrai projet en production avec des documents français, tu opterais pour des modèles encore plus puissants comme camembert (spécialisé français) ou Gemini Embedding (multilingue et très performant, mais payant).

la comparaison V1 vs V2, les différences sont parlantes !

## Les résultats sont très éloquents ! Analysons ce qui se passe.

### Ce que les résultats nous montrent
La pertinence du V2 est nettement meilleure

Regarde les améliorations concrètes. Pour "budget de l'éclairage public", V1 classait le bon document en 3ème position, V2 le place en 1er, 2ème ET 3ème — il a compris que la question porte sur l'éclairage, pas juste sur un budget quelconque. Pour "le règlement municipal concernant le bruit", V1 mettait un projet d'éclairage en 1er (faux positif flagrant), V2 place correctement le RÈGLEMENT MUNICIPAL.md aux trois premières positions. Pour le marché de Noël, V2 retrouve aussi l'image OCR du marché de Noël (marche_noel_trifouillis_2023.md) en 3ème position, que V1 ne trouvait même pas.
La preuve chiffrée avec la similarité

C'est le résultat le plus parlant de tout l'exercice :
```
"budget de l'éclairage public" vs "financement des lampadaires de la commune"
   V1 (anglais)     : 0.3714   ← Il ne voit presque pas le lien !
   V2 (multilingue) : 0.8130   ← Il comprend que c'est le même sujet

"budget de l'éclairage public" vs "marché de Noël de la ville"
   V1 (anglais)     : 0.3868   ← Il met ça au MÊME niveau que le précédent !
   V2 (multilingue) : 0.2998   ← Il distingue bien que c'est un autre sujet
  ```

V1 donne quasiment le même score (0.37 vs 0.39) à deux phrases du même thème et à deux phrases de thèmes différents. C'est catastrophique : ça veut dire qu'il ne comprend pas vraiment le sens en français et se repose sur des indices superficiels. V2 en revanche fait un écart net (0.81 vs 0.30), il discrimine correctement.
Pourquoi les distances V2 sont-elles si grandes (6, 10, 18...) ?

Tu as dû remarquer que les distances V1 sont entre 0 et 1, alors que les distances V2 montent à 10, 18, 25... Pas de panique, ce n'est pas un bug. C'est simplement parce que ChromaDB utilise par défaut la distance L2 (euclidienne) et non la similarité cosinus. Les deux modèles ont des distributions de vecteurs différentes, donc les échelles ne sont pas directement comparables. Ce qui compte, c'est le classement (quel document arrive en premier), pas la valeur absolue de la distance.

##Bilan : quel modèle garder ?

On garde évidemment V2 (multilingue) pour la suite. Les résultats montrent clairement qu'avec des documents en français, le choix du modèle d'embedding n'est pas un détail — c'est un facteur déterminant pour la qualité du RAG.






# Connecter un LLM à notre base vectorielle pour créer le pipeline RAG complet

## Comprendre le pipeline RAG complet
Ce qu'on a construit jusqu'ici
```

Documents bruts (PDF, DOCX, PNG, CSV...)
        │
        ▼
   ① DOCLING → Markdown                    ✅ Fait
        │
        ▼
   ② CHUNKING → 195 chunks                 ✅ Fait
        │
        ▼
   ③ EMBEDDING → Vecteurs dans ChromaDB     ✅ Fait
        │
        ▼
   ④ RECHERCHE + LLM → Réponse             🔜 C'est maintenant !

   ```

### Comment fonctionne l'étape 4 ?

Quand l'utilisateur pose une question, le système fait trois choses dans l'ordre.
- D'abord, il recherche les chunks les plus pertinents dans ChromaDB (ce qu'on a déjà testé).
- Ensuite, il construit un prompt qui contient la question de l'utilisateur ET les chunks retrouvés comme contexte.
- Enfin, il envoie ce prompt au LLM qui génère une réponse en se basant sur les documents fournis, pas sur ses connaissances générales.

C'est ça la magie du RAG : le LLM ne répond pas "de mémoire", il répond à partir des documents de Triffouillis sur Loire. Il peut citer des montants précis, des dates, des noms, parce qu'il a les extraits pertinents sous les yeux.

### Le prompt : l'ingrédient secret

Le prompt qu'on envoie au LLM ressemble à ceci :

Tu es un assistant de la mairie de Triffouillis sur Loire.
Réponds à la question en te basant UNIQUEMENT sur les documents fournis.
Si l'information n'est pas dans les documents, dis-le clairement.

--- Documents pertinents ---
[Chunk 1 : extrait du projet d'éclairage...]
[Chunk 2 : extrait du budget...]
[Chunk 3 : ...]

--- Question ---
Quel est le budget prévu pour l'éclairage public ?

C'est ce qu'on appelle le grounding : on ancre le LLM dans des faits concrets. Ça réduit drastiquement les hallucinations.


### Le LLM qu'on va utiliser : Hugging Face (open-source, gratuit)

Pour rester dans le cadre pédagogique et open-source, on va utiliser l'API d'inférence gratuite de Hugging Face avec le modèle Mistral-7B-Instruct.

C'est un excellent LLM open-source qui comprend très bien le français. Il te faudra juste un token Hugging Face gratuit.

In [19]:
# On va normaliser les distances
# Créer la collection V2 avec la métrique cosinus
try:
    chroma_client.delete_collection("documents_municipaux_v2_cosine")
except:
    pass

collection_v2_cosine = chroma_client.create_collection(
    name="documents_municipaux_v2_cosine",
    metadata={"hnsw:space": "cosine"}
)

collection_v2_cosine.add(
    ids=ids,
    embeddings=embeddings_v2.tolist(),
    documents=texts,
    metadatas=metadatas
)

print(f"✅ Collection V2 cosine créée avec {collection_v2_cosine.count()} chunks !")

✅ Collection V2 cosine créée avec 195 chunks !


In [20]:
#Installer le client Mistral
!pip install --upgrade mistralai



  Using cached opentelemetry_semantic_conventions-0.60b1-py3-none-any.whl.metadata (2.4 kB)
  Using cached opentelemetry_api-1.39.1-py3-none-any.whl.metadata (1.5 kB)
Using cached opentelemetry_semantic_conventions-0.60b1-py3-none-any.whl (219 kB)
Using cached opentelemetry_api-1.39.1-py3-none-any.whl (66 kB)
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.40.0
    Uninstalling opentelemetry-api-1.40.0:
      Successfully uninstalled opentelemetry-api-1.40.0
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.61b0
    Uninstalling opentelemetry-semantic-conventions-0.61b0:
      Successfully uninstalled opentelemetry-semantic-conventions-0.61b0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-sdk 1.40.0 requires opentelemetry-

In [21]:
#Configurer le token et tester

from google.colab import userdata
import os

# Option A : Via les secrets Colab (icône 🔑 à gauche > ajouter "MISTRAL_API_KEY")
try:
    MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')
    print("✅ Clé récupérée depuis les secrets Colab")
except:
    # Option B : Saisie directe
    MISTRAL_API_KEY = input("🔑 Colle ta clé API Mistral ici : ")
    print("✅ Clé configurée")

✅ Clé récupérée depuis les secrets Colab


In [22]:
import mistralai.client

print(f"Content of mistralai.client module: {dir(mistralai.client)}")

# If MistralClient is present, try to import it.
if 'MistralClient' in dir(mistralai.client):
    from mistralai.client import MistralClient
    print("MistralClient imported successfully after inspection.")
else:
    print("MistralClient not found in mistralai.client. Please check mistralai package version or restart runtime.")
    # If MistralClient is still not found, it might be necessary to restart the runtime and reinstall.
    # from mistralai.client import MistralClient # Uncomment and retry after restarting runtime if needed

Content of mistralai.client module: ['Any', 'AsyncHttpClient', 'BaseSDK', 'Callable', 'ClientOwner', 'Dict', 'Field', 'HttpClient', 'Logger', 'Mistral', 'OPENAPI_DOC_VERSION', 'Optional', 'OptionalNullable', 'RetryConfig', 'SDKConfiguration', 'SDKHooks', 'SERVERS', 'SERVER_EU', 'SPEAKEASY_GENERATOR_VERSION', 'TYPE_CHECKING', 'Tuple', 'UNSET', 'USER_AGENT', 'Union', 'VERSION', '__annotations__', '__builtins__', '__cached__', '__doc__', '__file__', '__gen_version__', '__loader__', '__name__', '__openapi_doc_version__', '__package__', '__path__', '__spec__', '__title__', '__user_agent__', '__version__', '_hooks', '_version', 'basesdk', 'cast', 'close_clients', 'dataclass', 'errors', 'get_default_logger', 'httpclient', 'httpx', 'importlib', 'models', 'models_', 'remove_suffix', 'sdk', 'sdkconfiguration', 'sys', 'types', 'utils', 'weakref']
MistralClient not found in mistralai.client. Please check mistralai package version or restart runtime.


In [23]:
# Cherchons le bon client
import importlib
import pkgutil

for importer, modname, ispkg in pkgutil.walk_packages(mistralai.__path__, prefix="mistralai."):
    print(modname)


mistralai.client
mistralai.client._hooks
mistralai.client._hooks.custom_user_agent
mistralai.client._hooks.deprecation_warning
mistralai.client._hooks.registration
mistralai.client._hooks.sdkhooks
mistralai.client._hooks.tracing
mistralai.client._hooks.types
mistralai.client._version
mistralai.client.accesses
mistralai.client.agents
mistralai.client.audio
mistralai.client.basesdk
mistralai.client.batch
mistralai.client.batch_jobs
mistralai.client.beta
mistralai.client.beta_agents
mistralai.client.campaigns
mistralai.client.chat
mistralai.client.chat_completion_events
mistralai.client.classifiers
mistralai.client.connectors
mistralai.client.conversations
mistralai.client.datasets
mistralai.client.documents
mistralai.client.embeddings
mistralai.client.errors
mistralai.client.errors.httpvalidationerror
mistralai.client.errors.mistralerror
mistralai.client.errors.no_response_error
mistralai.client.errors.observabilityerror
mistralai.client.errors.responsevalidationerror
mistralai.client.er

In [24]:
import requests
import json

MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')

def appel_mistral(messages, max_tokens=1024, temperature=0.3):
    """
    Appel direct à l'API REST Mistral — aucune dépendance de bibliothèque
    """
    response = requests.post(
        "https://api.mistral.ai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {MISTRAL_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": "mistral-small-latest",
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": temperature
        }
    )

    if response.status_code != 200:
        raise Exception(f"Erreur Mistral API ({response.status_code}): {response.text}")

    return response.json()["choices"][0]["message"]["content"]

# --- Test rapide ---
test = appel_mistral([{"role": "user", "content": "Dis bonjour en français en une phrase."}])
print(f"✅ Mistral connecté ! Test : {test}")


✅ Mistral connecté ! Test : Bonjour ! 😊


In [25]:
#Lancement du RAG

def recherche_rag(question, n_results=5):
    """
    Recherche les chunks pertinents dans ChromaDB
    """
    query_embedding = embedding_model_v2.encode([question]).tolist()

    results = collection_v2_cosine.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )

    contexte_chunks = []
    sources = []

    for i in range(len(results['documents'][0])):
        doc = results['documents'][0][i]
        meta = results['metadatas'][0][i]
        dist = results['distances'][0][i]

        contexte_chunks.append(
            f"[Document : {meta['source']} | Catégorie : {meta['category']}]\n{doc}"
        )
        sources.append({
            "source": meta['source'],
            "category": meta['category'],
            "distance": dist
        })

    contexte = "\n\n---\n\n".join(contexte_chunks)
    return contexte, sources


def generer_reponse_rag(question, n_results=5):
    """
    Pipeline RAG complet : recherche + génération avec Mistral
    """
    contexte, sources = recherche_rag(question, n_results)

    messages = [
        {
            "role": "system",
            "content": """Tu es un assistant municipal de la Mairie de Triffouillis sur Loire.
Tu réponds aux questions des citoyens en te basant UNIQUEMENT sur les documents municipaux fournis.

Règles importantes :
- Réponds en français, de manière claire et professionnelle.
- Cite les sources (noms des documents) quand tu donnes une information.
- Si l'information demandée n'est pas dans les documents fournis, dis-le honnêtement.
- Ne fais pas d'invention. Reste fidèle aux documents."""
        },
        {
            "role": "user",
            "content": f"""Voici les documents municipaux pertinents :

{contexte}

--- QUESTION ---

{question}"""
        }
    ]

    reponse_texte = appel_mistral(messages, max_tokens=1024, temperature=0.3)
    return reponse_texte, sources


print("✅ Fonctions RAG avec Mistral prêtes !")


✅ Fonctions RAG avec Mistral prêtes !


In [26]:
# Test du RAG

question = "Quel est le budget prévu pour l'éclairage public ?"

print(f"🔍 Question : \"{question}\"\n")
print("⏳ Recherche et génération en cours...\n")

reponse, sources = generer_reponse_rag(question)

print(f"{'='*60}")
print(f"🤖 RÉPONSE DU RAG :")
print(f"{'='*60}")
print(reponse)

print(f"\n{'='*60}")
print(f"📚 SOURCES UTILISÉES :")
print(f"{'='*60}")
for i, src in enumerate(sources):
    print(f"   {i+1}. [{src['category']}] {src['source']} "
          f"(distance: {src['distance']:.4f})")


🔍 Question : "Quel est le budget prévu pour l'éclairage public ?"

⏳ Recherche et génération en cours...

🤖 RÉPONSE DU RAG :
D'après les documents municipaux fournis, voici les informations concernant le budget prévu pour l'éclairage public :

1. **Projet de Modernisation de l'Éclairage Public** :
   - Budget alloué : **300 000 €** (validé lors de la réunion du 5 octobre 2023 - *PV_05102023.md*).
   - Ce budget couvre le remplacement des infrastructures lumineuses obsolètes par des systèmes LED intelligents.

2. **Développement de nouvelles pistes cyclables et zones piétonnes (2026)** :
   - Renforcement de l'éclairage public LED sur 3 sections de pistes cyclables : **20 000 €** (T4 2026).

**Sources :**
- *PV_05102023.md* (Point 2 - Modernisation de l'éclairage public).
- *Développement de nouvelles pistes cyclables et zones piétonnes (2026).md* (Section 3.2 Éclairage Abords Pistes).

Si vous cherchez un budget global incluant d'autres projets connexes, merci de préciser votre demande

Analyse de cette réponse
La pertinence est au rendez-vous

Le RAG a trouvé 300 000 € pour le projet d'éclairage public ET 20 000 € supplémentaires pour l'éclairage des pistes cyclables. Il ne s'est pas contenté d'un seul document, il a croisé les sources pour donner une réponse complète. C'est exactement le comportement qu'on veut.
Les sources sont citées correctement

Mistral a respecté nos instructions du prompt : il cite les noms des documents, les sections précises, et propose même d'affiner la recherche si besoin. C'est professionnel et transparent.
Les distances sont excellentes

Compare avec ce qu'on avait avant :
```
AVANT (V1 anglais) :
   1. Suivi Centre Culturel     → distance: 0.7856  (MAUVAIS document)
   2. Pistes cyclables           → distance: 0.7973  (MAUVAIS document)
   3. Éclairage public           → distance: 0.7993  (bon, mais 3ème...)

MAINTENANT (V2 multilingue + cosine) :
   1. Éclairage public           → distance: 0.2492  ✅
   2. Pistes cyclables (éclairage)→ distance: 0.3065  ✅
   3. Éclairage public           → distance: 0.3085  ✅
```

On est passé de distances autour de 0.8 à des distances autour de 0.25-0.33. C'est une amélioration massive, et surtout le bon document est en premier.

In [27]:
questions = [
    "Quand a lieu le marché de Noël et quelles sont les activités prévues ?",
    "Comment signaler un problème d'éclairage public dans ma rue ?",
    "Quels sont les horaires d'ouverture de la mairie ?",
    "Quels projets de développement durable sont prévus par la commune ?",
    "Qui est le maire de Triffouillis sur Loire ?",
    "Quel est le budget total des projets prévus pour 2024 ?",
]

for question in questions:
    print(f"\n{'='*70}")
    print(f"🔍 QUESTION : {question}")
    print(f"{'='*70}\n")

    reponse, sources = generer_reponse_rag(question)

    print(f"🤖 RÉPONSE :")
    print(reponse)

    print(f"\n📚 Sources :")
    for i, src in enumerate(sources[:3]):
        print(f"   {i+1}. [{src['category']}] {src['source']} (dist: {src['distance']:.4f})")
    print()



🔍 QUESTION : Quand a lieu le marché de Noël et quelles sont les activités prévues ?

🤖 RÉPONSE :
D'après le **Bulletin Municipal – Marché de Noël 2023**, voici les informations disponibles :

**Date du marché de Noël :**
Les **16 et 17 décembre 2023** sur la Place du Grand Chêne à Triffouillis-sur-Loire.

**Activités prévues :**
Le document mentionne une **ambiance chaleureuse garantie pour petits et grands**, mais **aucune liste détaillée des animations spécifiques** n'est fournie dans le texte disponible. Pour obtenir des précisions sur les activités, je vous invite à consulter le site internet de la mairie ou à les contacter directement, comme indiqué dans le document **Accueil_affichage_Mairie de Triffouillis sur Loire**.

📚 Sources :
   1. [intances] Bulletin Municipal – Marché de Noël 2023.md (dist: 0.4027)
   2. [communication] Acceuil_affichage_Mairie de Triffouillis sur Loire.md (dist: 0.5224)
   3. [intances] PV_20122023.md (dist: 0.5622)


🔍 QUESTION : Comment signaler un p

## Ce qui fait la différence

Ce n'est pas Mistral seul qui produit ces résultats. Un LLM sans RAG aurait inventé des chiffres et des noms. C'est la combinaison de toutes les briques qu'on a construites ensemble qui fait la magie :
- Docling pour extraire proprement le contenu,
- un chunking intelligent avec overlap,
- un modèle d'embedding multilingue qui comprend le français, - ChromaDB avec la bonne métrique,

et un prompt système bien cadré qui force Mistral à rester fidèle aux sources.

In [28]:
# ============================================================
# 🏛️  CHATBOT RAG — MAIRIE DE TRIFFOUILLIS SUR LOIRE
# ============================================================

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# --- Interface stylée ---
display(HTML("""
<div style="background: linear-gradient(135deg, #1e3a5f, #2d6a9f);
            padding: 20px; border-radius: 15px; margin-bottom: 20px;
            color: white; text-align: center;">
    <h1 style="margin: 0;">🏛️ Assistant Municipal</h1>
    <h3 style="margin: 5px 0; font-weight: normal;">
        Mairie de Triffouillis sur Loire
    </h3>
    <p style="margin: 5px 0; opacity: 0.8;">
        Posez vos questions sur la commune — Tapez <b>quit</b> pour quitter
    </p>
</div>
"""))

# --- Historique de conversation ---
historique = []

def afficher_message(role, texte, sources=None):
    if role == "user":
        display(HTML(f"""
        <div style="background: #e3f2fd; padding: 12px 16px; border-radius: 12px;
                    margin: 8px 0; margin-left: 60px; border-left: 4px solid #1976d2;">
            <b>🧑 Vous :</b><br>{texte}
        </div>
        """))
    else:
        sources_html = ""
        if sources:
            sources_html = "<br><div style='margin-top:10px; padding-top:8px; border-top:1px solid #ddd;'>"
            sources_html += "<b>📚 Sources :</b><br>"
            for i, src in enumerate(sources[:3]):
                dist = src['distance']
                # Couleur selon la pertinence
                color = "#4caf50" if dist < 0.3 else "#ff9800" if dist < 0.5 else "#f44336"
                sources_html += (
                    f"&nbsp;&nbsp;{i+1}. [{src['category']}] {src['source']} "
                    f"<span style='color:{color}; font-weight:bold;'>"
                    f"(pertinence: {(1-dist)*100:.0f}%)</span><br>"
                )
            sources_html += "</div>"

        display(HTML(f"""
        <div style="background: #f5f5f5; padding: 12px 16px; border-radius: 12px;
                    margin: 8px 0; margin-right: 60px; border-left: 4px solid #4caf50;">
            <b>🤖 Assistant :</b><br>{texte.replace(chr(10), '<br>')}
            {sources_html}
        </div>
        """))

# --- Boucle de conversation ---
print("💡 Exemples de questions :")
print("   • Quel est le budget de l'éclairage public ?")
print("   • Qui est le maire ?")
print("   • Quels événements sont prévus ?")
print("   • Quels sont les horaires de la mairie ?")
print()

while True:
    question = input("🧑 Votre question : ")

    if question.lower().strip() in ['quit', 'exit', 'q', 'quitter']:
        display(HTML("""
        <div style="background: #fff3e0; padding: 15px; border-radius: 12px;
                    text-align: center; margin: 10px 0;">
            <h3>👋 Merci de votre visite à la Mairie de Triffouillis sur Loire !</h3>
            <p>N'hésitez pas à revenir pour toute question.</p>
        </div>
        """))
        break

    if not question.strip():
        continue

    # Afficher la question
    afficher_message("user", question)

    try:
        # Appel au RAG
        reponse, sources = generer_reponse_rag(question)

        # Afficher la réponse
        afficher_message("assistant", reponse, sources)

        # Sauvegarder dans l'historique
        historique.append({
            "question": question,
            "reponse": reponse,
            "sources": sources
        })

    except Exception as e:
        display(HTML(f"""
        <div style="background: #ffebee; padding: 12px; border-radius: 12px;
                    margin: 8px 0; border-left: 4px solid #f44336;">
            <b>❌ Erreur :</b> {str(e)}<br>
            Réessayez avec une autre formulation.
        </div>
        """))

    print()  # Espacement entre les échanges


💡 Exemples de questions :
   • Quel est le budget de l'éclairage public ?
   • Qui est le maire ?
   • Quels événements sont prévus ?
   • Quels sont les horaires de la mairie ?

🧑 Votre question : Quels évènements pour les jeunes ados sont prévues ?



🧑 Votre question : quel est le nombre d'habitant dans la commune ? un recencement est-il d'actualité ?



🧑 Votre question : quit
